# StruQ Defense: Step-by-Step Structured Query Fine-Tuning & Evaluation

This notebook demonstrates how to build, fine-tune, load, and evaluate **StruQ** (*StruQ: Defending Against Prompt Injection with Structured Queries*, Chen et al., USENIX Security 2025) within the `ipi` benchmark framework.

## 📌 Overview & Workflow
1. **Environment & Dependency Setup**: Install `ipi` framework, HuggingFace `transformers`, `peft`, `trl`, `datasets`, and `bitsandbytes`.
2. **Dataset Setup & Anti-Instruction Data Construction**: Download clean instruction dataset (`alpaca_data_cleaned.json`) directly in notebook and generate StruQ anti-instruction tuning data with structured delimiters (`[MARK] [INST] [COLN]` vs `[MARK] [INPT] [COLN]`).
3. **Tokenizer Resizing & Embedding Warm-Start**: Enlarge vocabulary with 5 special delimiter tokens (`[INST]`, `[INPT]`, `[RESP]`, `[MARK]`, `[COLN]`) and initialize their embeddings with corresponding textual token embeddings (`instruction`, `input`, `response`, `###`, `:`).
4. **Supervised Fine-Tuning (SFT / QLoRA)**: Fine-tune model with `trl.SFTTrainer` and 4-bit NormalFloat quantization (BitsAndBytes) on Kaggle T4 GPU (~16GB VRAM) or full precision.
5. **Save, Reload & Export Model Adapters**: Save fine-tuned adapters to local disk and reload them into `LocalLLM`.
6. **Benchmark Evaluation**: Wrap reloaded model into `StruQDefense` and run comparative evaluations against `ipi` prompt injection attacks.

In [ ]:
# Cell 1 — Installation & Environment Setup
!pip install -q git+https://github.com/alirezaAalaie/IPI-Aaptive.git
!pip install -q trl peft transformers datasets bitsandbytes accelerate

# Kaggle ships a pinned torchao (e.g. 0.10.0) that is too old for the peft
# version installed above. peft eagerly version-checks torchao on import
# (even though StruQ's LoRA adapters don't use torchao at all) and raises
# "Found an incompatible version of torchao..." the moment PeftModel.from_pretrained()
# is called — which silently aborts adapter loading in LocalLLM. Since we don't
# need torchao here, remove it so peft's is_torchao_available() check is skipped.
!pip uninstall -y -q torchao

import os
import json
import urllib.request
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2 — Dataset Setup & StruQ Anti-Instruction Data Construction (Run Directly in Notebook)
os.makedirs("data", exist_ok=True)
DATA_PATH = "data/alpaca_data_cleaned.json"
DATA_URL = "https://raw.githubusercontent.com/gururise/AlpacaDataCleaned/refs/heads/main/alpaca_data_cleaned.json"

if not os.path.exists(DATA_PATH):
    print(f"Downloading dataset from {DATA_URL}...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"✅ Downloaded {DATA_PATH} ({os.path.getsize(DATA_PATH)} bytes)")
else:
    print(f"✅ Using existing dataset at {DATA_PATH}")

# Load clean samples
with open(DATA_PATH, "r") as f:
    clean_samples = json.load(f)

print(f"Loaded {len(clean_samples)} clean instruction samples.")

from ipi.defenses.struq import generate_struq_training_data, format_struq_prompt, STRUQ_DELIMITERS

# Generate StruQ anti-instruction fine-tuning dataset with SpclSpclSpcl delimiters
struq_training_data = generate_struq_training_data(
    clean_samples=clean_samples[:1000],   # Use subset for quick experiment or full dataset
    attack_type="NaiveCompletion",
    delimiter_scheme="SpclSpclSpcl",
    seed=42
)

print(f"✅ Generated {len(struq_training_data)} StruQ training samples!")
print("\n--- SAMPLE STRUCTURED QUERY (INJECTED INPUT) ---")
print(struq_training_data[1]["prompt"])
print("--- TARGET OUTPUT (CLEAN TASK ONLY) ---")
print(struq_training_data[1]["output"])


In [ ]:
# Cell 3 — Tokenizer Special Token Resizing & Embedding Warm-Start
import transformers
from ipi.defenses.struq import smart_tokenizer_and_embedding_resize, SPECIAL_DELM_TOKENS

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./struq_lora_weights"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Special delimiter tokens to add:", SPECIAL_DELM_TOKENS)
# smart_tokenizer_and_embedding_resize is automatically invoked during train_struq()


In [ ]:
# Cell 4 — Supervised Fine-Tuning (SFT / QLoRA with Paper Hyperparameters)
from ipi.defenses.struq import train_struq

# Fine-tuning on Kaggle T4 GPU (16 GB VRAM) or local GPU:
#  - 1.5B - 3B Models (Qwen2.5-1.5B, Llama-3.2-3B): ~15-30 mins training time
#  - 7B - 8B Models (Mistral-7B-v0.1, Llama-3-8B): ~1-2 hours training time with 4-bit QLoRA

"""
# Launch fine-tuning step-by-step (Uncomment to execute full fine-tuning on GPU kernel)
trainer = train_struq(
    model_name_or_path=MODEL_ID,
    output_dir=OUTPUT_DIR,
    train_samples=clean_samples[:1000],
    attack_type="NaiveCompletion",
    delimiter_scheme="SpclSpclSpcl",
    use_4bit=True,                   # Enables 4-bit NF4 QLoRA for low VRAM
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=512
)
print(f"✅ Training complete! Adapter saved to {OUTPUT_DIR}")
"""
print("Fine-tuning configuration ready.")


In [ ]:
# Cell 5 — Save, Export & Reload Model Adapters
"""
# Step 1: Save / Reload Adapter into LocalLLM
from ipi.llm_unified import LocalLLM

reloaded_victim = LocalLLM(
    model_name_or_path=MODEL_ID,
    adapter_path=OUTPUT_DIR
)
print("✅ Successfully reloaded StruQ fine-tuned adapter!")

# Step 2 (Optional): Push Adapter to Hugging Face Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="your-username/StruQ-Qwen2.5-1.5B-LoRA",
    repo_type="model"
)
"""
print("Adapter saving, reloading, and publishing script ready.")


In [ ]:
# Cell 6 — Evaluate StruQ Defense on ipi Attack Benchmark
from ipi.target import LocalLLM, TargetLLM
from ipi.defenses.struq import StruQDefense
from ipi.attacks import NaiveAttacker, IgnoreAttacker, FakeCompletionAttacker, EscapeAttacker
from ipi.evaluator import BipiaSuccessEvaluator, AttackEvaluator
from ipi.dataset import HijackDataset, DualVerifiableDataset

"""
# 1. Initialize StruQ defended target
base_llm = LocalLLM(model_name_or_path=MODEL_ID)
struq_target = StruQDefense(
    target=TargetLLM(base_llm.with_config(temperature=0.0, max_tokens=500)),
    delimiter_scheme="SpclSpclSpcl",
    use_struq_template=True
)

# 2. Select benchmark dataset
dataset = DualVerifiableDataset()
subset = dataset.subset(10, seed=42)

# 3. Run evaluation across static attacks
attackers = [
    NaiveAttacker(),
    EscapeAttacker(),
    IgnoreAttacker(),
    FakeCompletionAttacker()
]

for attacker in attackers:
    evaluator = AttackEvaluator(target=struq_target, attacker=attacker)
    eval_result = evaluator.run(subset)
    print(f"Attack: {attacker.__class__.__name__:<22} ASR: {eval_result.attack_success_rate * 100:.1f}%")
"""
print("StruQ evaluation pipeline configured.")
